<div align=center>

---
# **Equilibrage des flux injectés**
---

</div>

L'injection des faisceaux lasers dans le composant photonique peut être contrôlé par un miroir déformable segmenté (1 segment = 1 faisceau). En déviant le faisseau légèrement hors-axe, on peut ainsi ajuster le flux lumineux injecté. On souhaite ici équilibrer les flux de sorte à ce qu'aucun des faisceaux injecté ne domine le résultat les opérations interférométriques.

Pour ce faire, on balaye en tip et tilt chaque segment inidividuellement de sorte à créer une carte d'injection :

`files/calibration_bal_maps.png`
![](files/calibration_bal_maps.png)

Via cette carte indiquant le flux injecté via le segement considéré (donc sur le canal d'injection associé) en fonction du tip/tilt appliqué, on peut récupérer facilement le pixel maximum, qui à défaut d'être l'exacte maximum d'injection, est ici satisfaisant pour nos applications car nous ne cherchons pas a maximiser le flux de photon au photon près. Une fois la position de tip/tilt maximisant l'injection trouvé, on regarde le segment qui injecte le moins et on l'utilise comme référence pour équiliber les flux injectés via les autres segments. Pour ce faire, on applique un algorithme de dichotomie classique sur l'axe de tilt avec un tip fixé au niveau du tip qui maximise l'injection. Ce choix permet de dévier le faiseau hors-axe perpendiculairement au plan d'injection dans le composant photonique et réduit ainsi les éventuelles contaminations.

`files/calibration_bal_dichotomy.png`
![](files/calibration_bal_dichotomy.png)

Un algorithme de dichotomie a une complexité en log2(N), ce qui permet d'effectuer l'équivalent d'un balayage de 1000 points en seulement 10 mesures. Une mesure prend entre 1 et 2 centième de seconde en tenant compte du temps d'intégration, du temps de calcul (temps négligeable) et d'un temps de stabilisation du miroir déformable avec des grandes marges de sécurité. Cet  algorithme permet donc un équilibrage rapide des flux avec un niveau de précision choisi. En ayant fixé le nombre d'itération maximales a 50, on obtient un flux équilibré à moins de 1% d'écart relatif.

`files/calibration_bal_comparison.png`
![](files/calibration_bal_comparison.png)

<div align=center>

---
# **Cross talk & variation de flux**
---

</div>

Des effets de cross-talk (contamination lumineuse entre canaux voisins) avaient été observés durant la thèse de Peter Chingaipe. La contribution de Marc-Antoine dans le remontage du banc optique a permis de grandement réduire considérablement ces effets, sans pour autant pouvoir les annuler complètement. je partage l'intuition de Frantz que ces effets de cross-talk ont lieu, de façon dominante, avant l'injection et que le cross-talk interne au composant sont négligeable.

Grâce aux retardateurs de phase présents directement sur le composant, on peut scanner en phase chaque entrée et faire apparaitre les franges interférométriques (position et visibilité), ce qui permet également de révéler des potentiels effets de cross-talk.

En utilisant ce même algorithme, il arrive parfois que ces effets soient particulièrement faibles comme sur les scan suivant. Ces résultats semblent aller dans le sens de l'intuition que les effets de cross-talk interne au composant (et qui devraient donc être systématiquement présent du à la nature particulièrement stable des composants photoniques) sont très faibles.

1 entrée active\
`files/systematic_scan_1_inputs_input.png`
![](files/systematic_scan_1_inputs_input.png)

On remarque sur ces scan a une seule entrée active que les modulations d'intensité en sortie du composant, essentiellement dûe aux effets de cross-talk et/ou de mauvaise extinction des autres canaux, sont de l'ordre d'une dizaine d'ADU. Il semble toutefois y avoir une exception lorsque seule l'entrée 0 est active, où les effets de cross-talk semblent très proche du bruit. Etant donné que l'entrée 0 est celle qui se trouve à l'extérieur de la première courbe "S-band" et qu'aucun élément avant ça ne permettrait de réduire les cross-talk venant d'un faisceau plutôt qu'un autre, cela suggère qu'il y a bien un léger effet de cross-talk dans ce premier S-band.

Ces effets deviennent donc particulièrement faible lorsque l'on provoque des interférences volontaires avec plusieurs entrées actives comme ici a deux entrées :

2 entrées actives\
`files/systematic_scan_2_inputs_input.png`
![](files/systematic_scan_2_inputs_input.png)

<div align=center>

---
# **Modèle matriciel**
---

</div>

Ces scans utilisant les déphaseurs intégrés à la chip permettent ensuite d'ajuste un modèle matriciel décrivant notre système interferométrique. Ce modèle est définit comme suit :

$$
\vec{O} = \left| \mathbf{C_{\text{out}}} \cdot \mathbf{M} \cdot \mathbf{P} \cdot \mathbf{C_{\text{in}}} \cdot \vec{E} \right|^2
$$

Avec
- $\vec{E}$ un vecteur compelexe de 4 éléments représentant nos 4 champs électriques d'entrées dans l'approximation scalaire (amplitude et phase)
- $\mathbf{C_{\text{in}}}$ une matrice complexe 4x4 représentant les effets de cross-talk en amont du composant. En l'absence de cross-talk, cettre matrice devrait donc être la matrice identité.
- $\mathbf{M}$ représente la matrice complexe d'interaction du MMI, supposé parfait (ses paramètres sont fixé à l'avance et ne sont pas modifié pendant l'ajustement) et définit ci-dessous.
- $\mathbf{P}$ est une matriciel diagonale contenant les termes de phases injectés. Cette matrice est donc également parfaitement connue et ne necessite aucune ajustement, elle permet de lever (au moins en partie) les dégénérescence entre $\mathbf{C_{\text{in}}}$ et $\mathbf{C_{\text{out}}}$
- $\mathbf{C_{\text{out}}}$ une matrice complexe 4x4 représentant les effets de cross-talk ayant lieu après le composant ou bien directement au sein du composant lui même dans la mesure où on suppose la matrice $\mathbf{M}$ parfaite, ce qui a pour conséquence de reporter les imperfections réelles de $\mathbf{M}$ sur $\mathbf{C_{\text{out}}}$. Si le composant est parfait et en l'absence de cross-talk après ce dernier, cettre matrice devrait donc également être la matrice identité.
- $\vec{O}$ vecteur d'intensités en sortie du système

$$
\mathbf{M} = \begin{pmatrix}
1 & 1 & 1 & 1\\
1 & -i & i & -1\\
1 & i & -1 & -1\\
1 & -1 & -1 & 1
\end{pmatrix}
$$

L'ajustement de ce modèle par une méthode des moindres carré (avec une contrainte de non-création d'énergie) donne les résultats suivants, effectués à 1h44 d'interval :

In [35]:
import numpy as np
from scripts import latexio
model_18h52 = np.load("files/Model_18h52.npz")
model_20h36 = np.load("files/Model_20h36.npz")

# Crosstalk avant la chip
latexio.display_complex_matrix(model_18h52['Cin'], name=r"C_{in,18h52}")
latexio.display_complex_matrix(model_20h36['Cin'], name=r"C_{in,20h36}")
diff_Cin = model_20h36['Cin'] / model_18h52['Cin']
latexio.display_complex_matrix(diff_Cin, name=r"\Delta C_{in}")

# Crosstalk après la chip
latexio.display_complex_matrix(model_18h52['Cout'], name=r"C_{out,18h52}")
latexio.display_complex_matrix(model_20h36['Cout'], name=r"C_{out,20h36}")
diff_Cout = model_20h36['Cout'] / model_18h52['Cout']
latexio.display_complex_matrix(diff_Cout, name=r"\Delta C_{out}")

# Champs e.m. actifs
latexio.display_complex_vector(model_18h52['Ein'], name=r"E_{on,18h52}")
latexio.display_complex_vector(model_20h36['Ein'], name=r"E_{on,20h36}")
diff_Eon = model_20h36['Ein'] / model_18h52['Ein']
latexio.display_complex_vector(diff_Eon, name=r"\Delta E_{on}")

# Champs e.m. inactifs
latexio.display_complex_vector(model_18h52['Eout'], name=r"E_{off,18h52}")
latexio.display_complex_vector(model_20h36['Eout'], name=r"E_{off,20h36}")
diff_Eoff = model_20h36['Eout'] / model_18h52['Eout']
latexio.display_complex_vector(diff_Eoff, name=r"\Delta E_{off}")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [ ]:
# Fraction du flux du premier faisceau injecté dans les canaux voisins
e = np.array([1+0j, 0, 0, 0], dtype=complex)
cin = np.abs(model_18h52['Cin'] @ e)**2
cout = np.abs(model_18h52['Cout'] @ e)**2
print(r)
print(f"Mesuré (total): {15/575:.2g}") # visbilité des phrases / offset, mesuré à main levé (je m'intéresse surtout à l'ordre de grandeur) sur les plots d'interférence à 1 entrée active
print(f"Model Cin: {np.sum(cin[1:]):.2g}")
print(f"Model Cout: {np.sum(cout[1:]):.2g}")

[8.97809429e-01 2.47155944e-03 6.78965194e-04 7.60526071e-04]
Mesuré: 0.026
Model Cin: 0.0023
Model Cout: 0.0026


Ces résultats semblent suggérer plusieurs choses.

La première est que les effets de cross-talk modélisées sont très faible et semble évoluer avec le temps (semble uniquement, contredit plus bas dans la partie analyse des pahseurs). Pour $\mathbf{C_{\text{out}}}$, cette variation semble difficile à expliquer étant donné que le MMI et tout ce qui suit est intégré au composant (donc fixe et stable) puis réinjecté en espace libre et directement imagé, donc très peu d'interaction potentielles entre les canaux de sortie. Il semble donc plus probale de ces variations soient liés à des erreurs de mesure et/ou de finesse de l'ajustement, bien qu'on peut pas exclure un effet de dérive thermique (je n'ai malheureusement pas mesuré la température ambiante à ces deux instants). Il serait possible de s'en assurer en effectuant plusieurs scan et/ou ajustement du modèle a des intervalles de temps plus ou moins court pour voir si cela impacte les vairations (rend plus problable que les vairations soient réelles) ou non (rend plus probable que les vairations viennent d'erreur de mesure et/ou d'ajustement du model).

Deuxièmement, on remarque aussi que les vairations sur $\vec{E}$ sont particulièrement impaortantes. Cela indique qu'on a vraiment peu de stabilité au niveau du laser, aussi bien en amplitude qu'en phase, ce qui impact la stabilité des opérations interférométriques et donc la précision des mesures.

Toujours concernant, les flux d'entré, les valeur de flux éteints $\vec{E_{\text{off}}}$ trouvé après ajustement du modèle sont particulièrement forts comparé aux flux d'entré actifs $\vec{E_{\text{on}}}$ (un rapport, à la louche, de 400 pour 1 en terme d'intensité) tandis que sur les cartes d'injection on voit un rapport de près de 2500 pour 1. Le modèle surestime donc $\vec{E_{\text{off}}}$, sans doute coorrigeable via une étape de traitement (à discuter).

Enfin, le modèle matriciel ne semble pas suggérer d'effet de cross-talk plus faible sur le premier canal que sur les autres, effet qu'on observe ne pratique.

En bref, au regard des matrices elles mêmes, l'ajustement matriciel dans les conditions actuelles ne semble pas permettre de décrire correctement les effets de cross-talk.

<div align=center>

---
# **Calibration du Null**
---

</div>

En utilisant les déphaseurs intégré au composant photonique et en exploitant un algorithme d'optimisation inspiré de la dichotomie et des algorithmes génétiques (comparables à un marcheur aléatoire dans l'espace des paramètres pour lequel on refuse les pas qui ne vont pas dans le sens de l'optimisation), on optimise localement l'opération d'annulation interférométrique. Pour celà, on utilise comme métrique la somme des flux intégrés sur les sorties annulés, que l'on va chercher à minimiser.

`files/null_calibration.png`
![](files/null_calibration.png)
`files/outputs_before_calibration.png`
![](files/outputs_before_calibration.png)
`files/outputs_after_calibration.png`
![](files/outputs_after_calibration.png)

<div align=center>

---
# **Analyse des phaseurs**
---

</div>

Si on considère les champs d'entrée calibrés pour optimiser le null et qu'on les propage mathématiquement en utilisant le modèle matriciel précédemment ajusté, on remarque non seulement que la disposition des ces phaseurs coorrespond dans l'ensemble à l'opération qu'on s'attendait à avoir via la matrice de transfert du MMI $\mathbf{M}$, mais aussi qu'une structure plus fine des phaseurs est conservée malgré les variations des éléments des matrices :

Phaseurs à 18h52\
`files/phasors_calibrated_18h52.png`
![](files/phasors_calibrated_18h52.png)

Phaseurs à 20h36\
`files/phasors_calibrated_20h36.png`
![](files/phasors_calibrated_20h36.png)

Cette observation pousse naturellement à combiner les matrices pour voir si les variations ne viennent pas d'un dégénerescence dans le modèle qui provoque une contrainte trop faible sur chaque paramètre.

In [31]:
# Modèle complet 18h52
F_18h52 = model_18h52['Cout'] @ model_18h52['M'] @ model_18h52['Cin']
latexio.display_complex_matrix(F_18h52, name=r"F_{18h52}")

# Modèle complet à 20h36
F_20h36 = model_20h36['Cout'] @ model_20h36['M'] @ model_20h36['Cin']
latexio.display_complex_matrix(F_20h36, name=r"F_{20h36}")

latexio.display_complex_matrix(F_18h52 / F_20h36, name=r"\Delta F")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

On constate donc bel et bien que l'ajustement du modèle est donc relativement fiable dans son ensemble bien que chaque élément ne soit pas suffisament contraint à cause de dégénérescences. On a donc finalement peu d'évolution des effets de cross-talk, contrairement à ma remarque précédente. En revanche on retrouve le soucis du premier canal qui ne semble pas plus épargné par ces effets de cross-talk que les autres, ce qui peine à expliquer ce qu'on observe.

<div align=center>

---
# **Distributions vs perturbations**
---

</div>

En introduisant, via le miroir déformable présent en amont du composant photonique, des perturbations de phase, on peut ainsi visualiser l'imapact de potentielles perturbations atmosphériques sur la distribution des intensités produites en sortei dŭcomposant.

A 0nm de RMS, on constate que les sortes semblent pratiquement gaussienne ce qui s'explique par la seule présence de bruit gaussien lié au détecteur et au bruit de photon.

`files/distrib_0nm_rms.png`
![](files/distrib_0nm_rms.png)

`files/distrib_20nm_rms.png`
![](files/distrib_20nm_rms.png)

`files/distrib_100nm_rms.png`
![](files/distrib_100nm_rms.png)

Il faut toutefois pousser jusqu'à ~100nm RMS pour ne plus distinguer l'effet de ces bruits gaussien et retrouver des distributions familières (bruit instrumental important ? a discuter)

En effectuant ces mesures a 2 jours d'interval, j'ai également remarqué que la forme des distributions pouvait assez considérablement varier malgré que la pipeline de calibration de l'ensemble du banc et de l'acquisition des données n'ai pas changé, indiquant soit une dérive importante de certains éléments du banc, soit une instabilité passagère ayant eu lieu durant la pipeline d'acquisition des données présentant des distributions inatendues. D'autres executions de cette pipeline permettront de discriminer ces deux possibilités.


`files/distributions_vs_rms_normal.png`
![](files/distributions_vs_rms_normal.png)
`files/distributions_vs_rms_bizarre.png`
![](files/distributions_vs_rms_bizarre.png)